# 本机底盘控制：STM32 + 外置 IMU

电机控制与轮式里程计使用本机 ROS2 串口驱动。静态标定阶段用已知安装的参考 IMU 核对水平与静止状态，估计外置 IMU 的陀螺零偏和竖直轴；运行阶段只依赖本机设备。

将 `local_chassis.example.yaml` 复制到本地配置目录，填写串口、波特率和参考 IMU 安装值。标定不是完整外参：安装 yaw、平移、动态时间偏移和加速度计偏置仍未知。此输出仅供平面运动控制，不能替代三维 LIO 的 IMU。

In [ ]:
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'rsim').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assets = root / 'assets' / 'local-chassis'
assets.mkdir(parents=True, exist_ok=True)
os.environ['NOTEBOOK_ASSETS_ROOT'] = str(root / 'assets')
config = Path(os.environ.get('RSIM_CHASSIS_CONFIG', str(root / 'configs' / 'local_chassis.yaml')))

from rsim import Runtime
from rsim.config.local_chassis import load_local_chassis as build, calibrate_local_chassis as calibrate
config

首次使用或安装变化后，在静止、近水平地面采集参考数据。此步骤只发送零速；确认配置后将开关改为 True。

In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    calibration = await calibrate(config, seconds=10.)
    print(calibration.to_dict())

默认运行仅请求零距离、零角度。所有驱动随 Runtime 退出回收。`pose` 返回带坐标系标签的 graphmap Pose；ROS 类型不出现在调用侧。

In [ ]:
robot = build(config, motion_enabled=False)
async with Runtime(robot.control):
    pose = (await robot.pose.get(timeout=15)).data
    print(pose)
    await robot.control.move(0.)
    await robot.control.rotate(yaw_deg=0.)
    print((await robot.chassis.state.get(timeout=5)).data)

实际运动使用同一接口：`move(distance_m)` 正数前进、负数后退；`rotate(yaw_deg=...)` 或 `rotate(yaw_rad=...)` 正数左转。下面示例默认关闭。运行前需确认场地、线缆、急停以及独立障碍监测；这个基础控制组件不提供避障规划。

In [ ]:
ENABLE_MOTION = False
if ENABLE_MOTION:
    robot = build(config, motion_enabled=True)
    async with Runtime(robot.control):
        await robot.pose.get(timeout=15)
        try:
            await robot.control.rotate(yaw_deg=10., timeout=15.)
            await robot.control.rotate(yaw_deg=-10., timeout=15.)
            # await robot.control.move(0.1, timeout=15.)
        finally:
            await robot.control.stop()